# Macaron drawing styles
Tries a few illustration styles on the same dishes so one can be picked. Only a contact sheet is saved.

In [ ]:
!pip -q install --no-deps rembg pymatting onnxruntime flatbuffers coloredlogs humanfriendly 2>&1 | tail -2
!python -c "import diffusers" 2>/dev/null || pip -q install --no-deps diffusers 2>&1 | tail -2

In [ ]:
import torch, time
from PIL import Image, ImageDraw, ImageFont
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from rembg import remove, new_session
pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe = pipe.to("cuda"); pipe.vae.enable_slicing()
CUT = new_session("isnet-general-use")
NEG = "photo, photograph, photorealistic, realistic, text, watermark, logo, hands, person, chopsticks, table, tablecloth, blurry, lowres, deformed, messy, cluttered, dark background"
def gen(p, seed):
    g = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt=p, negative_prompt=NEG, width=1024, height=1024, num_inference_steps=28, guidance_scale=6.5, generator=g).images[0]
def cut(img):
    o = remove(img, session=CUT).convert("RGBA"); bb = o.getbbox()
    if bb: o = o.crop(bb)
    o.thumbnail((300, 300), Image.LANCZOS); return o

In [ ]:
DISHES = {
 "Nasi lemak": "Malaysian nasi lemak on a round white plate: a dome of white coconut rice, a spoon of red sambal, crispy fried anchovies and peanuts, cucumber slices, half a boiled egg",
 "Curry laksa": "Malaysian curry laksa in a white bowl: orange coconut curry soup, yellow noodles, tofu puffs, prawns, half a boiled egg, bean sprouts, mint",
 "Roti canai": "Malaysian roti canai: a round golden flaky flatbread folded on a plate with a small bowl of yellow dhal curry",
 "Char kuey teow": "Malaysian char kuey teow: smoky stir-fried flat rice noodles with prawns, bean sprouts, chives and egg on a white plate",
 "Chicken rice": "Malaysian Hainanese roasted chicken rice: sliced roasted chicken with glossy skin, a dome of rice, cucumber slices, chili sauce, on a white plate",
 "Wantan mee dry": "Malaysian dry wantan mee: springy egg noodles tossed in dark soy sauce, slices of red char siu pork, choy sum, wontons, on a white plate",
 "Fried wantan": "five golden crispy fried wontons with ruffled edges on a small white plate with sweet chili sauce",
 "Kaya toast set": "Malaysian kaya toast: two slices of toasted bread with green kaya jam and butter, with two soft boiled eggs in a small bowl",
}
STYLES = {
 "A soft painted": "a soft digital painting illustration of {d}, smooth painterly shading, gentle highlights, warm soft light, one cohesive illustration, cookbook illustration, three-quarter view, centered, plain white background",
 "B smooth vector": "a polished vector illustration of {d}, smooth gradients, soft shadows, clean rounded shapes, modern food app illustration, three-quarter view, centered, plain white background",
 "C 3D clay": "a cute 3D rendered icon of {d}, soft matte clay style like a modern 3D emoji, smooth studio lighting, three-quarter view, centered, plain white background",
 "D watercolour": "a delicate watercolour and fine ink illustration of {d}, soft washes, light pencil lines, three-quarter view, centered, plain white background",
}
res = {}
for d, desc in DISHES.items():
    for s, p in STYLES.items():
        t = time.time(); res[(d, s)] = cut(gen(p.format(d=desc), 17)); print(d, "|", s, round(time.time() - t), "s")

In [ ]:
C = 230; W = len(STYLES); H = len(DISHES)
sheet = Image.new("RGB", (160 + W * C, 50 + H * C), "#fff0f4"); dr = ImageDraw.Draw(sheet)
try: f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
except Exception: f = ImageFont.load_default()
for j, s in enumerate(STYLES): dr.text((170 + j * C, 14), s, fill="#7a2e4d", font=f)
for i, d in enumerate(DISHES):
    dr.text((10, 50 + i * C + C // 2 - 10), d, fill="#7a2e4d", font=f)
    for j, s in enumerate(STYLES):
        im = res[(d, s)].copy(); im.thumbnail((C - 16, C - 16)); x = 160 + j * C + (C - im.width) // 2; y = 50 + i * C + (C - im.height) // 2
        sheet.paste(im, (x, y), im)
sheet.save("/kaggle/working/contact_styles.jpg", quality=86); print("sheet saved", sheet.size)